In [2]:
! uv add tensorflow-datasets
! uv add importlib-resources

Resolved 98 packages in 8ms
Checked 92 packages in 28ms
Resolved 98 packages in 10ms
Checked 92 packages in 2ms


In [3]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "../src"))

In [4]:
import tensorflow_datasets as tfds
import numpy as np

# 1. Load the 'emnist/letters' dataset
# as_supervised=True gives us (image, label) pairs
ds_train, ds_info = tfds.load('emnist/letters', split='train', as_supervised=True, with_info=True)
ds_test = tfds.load('emnist/letters', split='test', as_supervised=True)

# 2. Convert to NumPy Arrays
# We use tfds.as_numpy to handle the conversion efficiently
train_data = list(tfds.as_numpy(ds_train))
test_data = list(tfds.as_numpy(ds_test))

# 3. Unpack and Preprocess
# Images come as (28, 28, 1). We need to flatten them to 784 and Transpose.
x_train = np.array([ex[0].flatten() for ex in train_data]).T / 255.0
y_train = np.array([ex[1] for ex in train_data])

x_test = np.array([ex[0].flatten() for ex in test_data]).T / 255.0
y_test = np.array([ex[1] for ex in test_data])

# 4. Correct the Label Offset
# TFDS EMNIST letters are 1-26. We need 0-25.
y_train = y_train - 1
y_test = y_test - 1

print(f"Dataset loaded successfully.")
print(f"Train Shape: {x_train.shape}, Labels: {np.unique(y_train)}")

I0000 00:00:1789909897.630491 28733316 tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Dataset loaded successfully.
Train Shape: (784, 88800), Labels: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25]


In [5]:
# Take transpose because the classifier expects (N, D)
x_train = x_train.T
x_test = x_test.T

In [6]:
from mlsuite.clf import OvALogisticClassification, LogisticClfConfig

conf = LogisticClfConfig(
    lr=0.001,
    l2_coef=1e-9,
    niters=10000,
    use_bias=True,
    num_classes=26
)

model = OvALogisticClassification(conf)

In [7]:
model.fit(x_train, y_train, np.float32)

In [ ]:
y_pred = model.predict(x_test) # (N, C)

y_pred = np.argmax(y_pred, axis=1)

In [20]:
acc = np.mean(y_pred == y_test)
acc

np.float64(0.5686486486486486)